1. Environment Setup & Data Loading

In [1]:
import pyspark

In [2]:
import os
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] += os.pathsep + r"C:\hadoop\bin"

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("big data").getOrCreate()

In [3]:
df=spark.read.parquet(r"C:\Users\HP TH\Downloads\yellow_tripdata_2026-01.parquet")

2. Exploratory Data Analysis (EDA)

In [4]:
df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2026-01-01 00:54:04|  2026-01-01 00:59:37|              1|         0.97|         1|                 N|         239|    

In [5]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [6]:
print(df.count())
print(len(df.columns))

3724889
20


3. Checking for Missing Values

In [7]:
from pyspark.sql.functions import col, count, when

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       0|                   0|                    0|        1088058|            0|   1088058|           1088058|           0|    

In [8]:
df.select([
    count(when(col(i).isNull(),i)).alias(i)
    for i in df.columns
]).show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       0|                   0|                    0|        1088058|            0|   1088058|           1088058|           0|    

In [9]:
df.select([
    count(when(col(i).isNull(),i)).alias(i)
    for i in df.columns
]).show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       0|                   0|                    0|        1088058|            0|   1088058|           1088058|           0|    

3. Data Cleaning & Filtering

In [10]:
df=df.dropDuplicates()

In [11]:
df = df.filter(col("trip_distance") > 0)

In [12]:
df=df.filter(col("fare_amount")>0)

4. Feature Engineering

In [13]:
from pyspark.sql.functions import unix_timestamp, round
df=df.withColumn(
    "trip_duration_minutes",round((unix_timestamp("tpep_dropoff_datetime")-unix_timestamp("tpep_pickup_datetime"))/60,2)
)

In [14]:
from pyspark.sql.functions import hour

df = df.withColumn(
    "pickup_hour",
    hour("tpep_pickup_datetime")
)

In [15]:
from pyspark.sql.functions import date_format
df=df.withColumn("pickup_day",date_format("tpep_pickup_datetime","EEE"))

5. Data Aggregation & Insights

In [16]:
from pyspark.sql.functions import avg,round
hourly_avg_fare=df.groupBy("pickup_hour").agg(round(avg("fare_amount"),2)).orderBy("pickup_hour")

In [17]:
from pyspark.sql.functions import desc
hourly_ride_count=df.groupBy("pickup_hour").count().orderBy("pickup_hour")

In [18]:
passenger_insights=df.groupBy("passenger_count").avg("trip_distance").orderBy("passenger_count")

Saving the cleaned data

In [19]:
output_file = r"C:\Users\HP TH\OneDrive\Desktop\NYC-Taxi-ETL\data\processed"
df.write.mode("overwrite").parquet(output_file)

hourly_avg_fare.write.mode("overwrite").parquet(r"C:\Users\HP TH\OneDrive\Desktop\NYC-Taxi-ETL\data\processed\hourly_avg_fare")
hourly_ride_count.write.mode("overwrite").parquet(r"C:\Users\HP TH\OneDrive\Desktop\NYC-Taxi-ETL\data\processed\hourly_ride_count")
passenger_insights.write.mode("overwrite").parquet(r"C:\Users\HP TH\OneDrive\Desktop\NYC-Taxi-ETL\data\processed\passenger_insights")

print("success")

success
